# Matriks Ablasi

Menggabungkan model A, B_1, B_2 (Tugas 2) dan semua varian C dari `results_c/`.

In [2]:
from c_common import *


## 1. Model A, B_1, B_2 (Tugas 2) dalam metrik yang sama dengan C

Skor dihitung ulang dari `predictions.csv` dengan `c_common.metrics`, jadi rumus metriknya identik dengan Model C.
- **A** = `LR/HGB [Harga]` (hanya time-series). Prediksinya sama di kedua file, jadi diambil dari `results/`.
- **B_1** = `[Harga+NLP]` pada `results/` (semua artikel). **B_2** = `[Harga+NLP]` pada `results_filtered/` (artikel terfilter).

In [3]:
def from_preds(path, label_fn, keep):
    """Satu baris per kolom model di predictions.csv, dengan skor val dan test."""
    P = pd.read_csv(path, parse_dates=["date"])
    rows = []
    for col in [c[7:] for c in P if c.startswith("score::") and keep(c[7:])]:
        r = {"name": label_fn(col), "model": col.split()[0]}
        for sp in ["val", "test"]:
            q = P[P.split == sp]
            r.update({f"{sp}_{k}": v for k, v in metrics(q.target, q[f"pred::{col}"], q[f"score::{col}"]).items()})
        rows.append(r)
    return pd.DataFrame(rows)

full_preds, filt_preds = DATA / "results" / "predictions.csv", DATA / "results_filtered" / "predictions.csv"
is_price = lambda c: c in ("LR [Harga]", "HGB [Harga]")
is_nlp = lambda c: c in ("LR [Harga+NLP]", "HGB [Harga+NLP]")
A  = from_preds(full_preds, lambda c: "A " + c.split()[0], is_price)
B1 = from_preds(full_preds, lambda c: "B_1 " + c.split()[0], is_nlp)
B2 = from_preds(filt_preds, lambda c: "B_2 " + c.split()[0], is_nlp)
AB = pd.concat([A, B1, B2], ignore_index=True)
AB

,name,model,val_DA,val_BalAcc,val_F1_macro,val_MCC,val_AUC,test_DA,test_BalAcc,test_F1_macro,test_MCC,test_AUC
0,A LR,LR,0.505682,0.508016,0.504386,0.016173,0.501228,0.516854,0.515469,0.512918,0.030561,0.526861
1,A HGB,HGB,0.522727,0.520621,0.519688,0.041543,0.502521,0.500000,0.502718,0.498083,0.005372,0.538123
2,B_1 LR,LR,0.517045,0.515902,0.515779,0.031859,0.473562,0.516854,0.517282,0.513847,0.034133,0.538511
3,B_1 HGB,HGB,0.517045,0.515514,0.515152,0.031139,0.471364,0.455056,0.456634,0.452967,-0.085700,0.490874
4,B_2 LR,LR,0.528409,0.530769,0.527173,0.062083,0.529282,0.500000,0.504531,0.498718,0.008963,0.561294
5,B_2 HGB,HGB,0.590909,0.591984,0.590856,0.184160,0.591726,0.505618,0.518447,0.505618,0.036893,0.506667


## 2. Model C terpilih: `C2_finbert_art_cls_HGB`

FinBERT, judul+lead **per artikel** (di-embed satu per satu, dirata-rata per hari), pooling `[CLS]`, PCA 20 komponen (di-fit hanya di train), HGB. Varian ini dipilih berdasarkan validation. Fitur dibangun ulang dari cache `c2_finbert_art_cls.npy` dengan langkah yang sama seperti `05b_C2_transformer`; sel pengecekan di bawah memastikan skor validation sama dengan yang tersimpan.

In [4]:
from sklearn.decomposition import PCA

C_NAME, C_KIND = "C2_finbert_art_cls_HGB", "HGB"
D, text = load_all()
TITLE, LEAD = "judul_gabungan", "lead_paragraph_gabungan"
day_text = (text[TITLE].fillna("") + ". " + text[LEAD].fillna("")).str.strip(" .")
has_news = (day_text.str.len() > 0).values

emb = pd.DataFrame(np.load(FEATURES_C / "c2_finbert_art_cls.npy"), index=text.index)
tr = D["train"]["date"]
tr = tr[has_news[text.index.get_indexer(tr)]]            # PCA hanya dari hari train yang punya teks
pca = PCA(n_components=20, random_state=SEED).fit(emb.loc[tr].values)
z = pca.transform(emb.values)
z[~has_news] = 0.0                                       # hari tanpa berita = 0 + penanda has
comp = pd.DataFrame(z, index=text.index, columns=[f"c2_art_cls_{i}" for i in range(20)])
comp["c2_art_cls_has"] = has_news.astype(float)
for s in D:
    D[s] = D[s].merge(comp, left_on="date", right_index=True, how="left")
C_COLS = PRICE + list(comp.columns)

rec = json.loads((RESULTS_C / f"{C_NAME}.json").read_text())
_, sc, _ = tune(C_KIND, D, C_COLS)
diff = max(abs(sc[k] - rec["val"][k]) for k in sc)
print(f"max selisih skor val vs results_c: {diff:.4f}")
if diff > 1e-9:
    import sklearn
    print(f"PERINGATAN: tidak identik (scikit-learn {sklearn.__version__}; hasil tersimpan dibuat dengan 1.8.0). "
          "Skor test di results_c tetap yang tersimpan; jangan jalankan ulang final_test di environment ini.")

max selisih skor val vs results_c: 0.0000


### Skor test (sekali saja)

`final_test` melatih ulang di train+val dengan hyperparameter terbaik dari validation, lalu menilai test. Fungsi ini menolak dipanggil dua kali untuk nama yang sama. Probabilitas per hari disimpan ke `results_c/preds/` untuk analisis error di `07_analysis`.

In [5]:
PRED_DIR = RESULTS_C / "preds"; PRED_DIR.mkdir(exist_ok=True)
if "test" not in json.loads((RESULTS_C / f"{C_NAME}.json").read_text()):
    prob = final_test(C_NAME, C_KIND, D, C_COLS)
    pd.DataFrame({"date": D["test"]["date"], "target": D["test"]["target"], "score": prob}).to_csv(PRED_DIR / f"{C_NAME}_test.csv", index=False)
else:
    print(C_NAME, "sudah dites; skor dibaca dari results_c.")

C2_finbert_art_cls_HGB sudah dites; skor dibaca dari results_c.


## 3. Matriks ablasi

Satu tabel untuk Model A, B_1, B_2 (Tugas 2), dan C (Tugas 3). Skor test C ditambah interval kepercayaan 95% AUC (block bootstrap).

In [6]:
crow = collect_results().query("name == @C_NAME").assign(name="C (FinBERT art-CLS)", model="HGB")
M = pd.concat([AB, crow], ignore_index=True)
cols = ["name", "model"] + [f"{s}_{m}" for s in ("val", "test") for m in ("DA", "BalAcc", "F1_macro", "MCC", "AUC")]
M = M[cols + ["test_auc_ci95"]]
M.to_csv(RESULTS_C / "ablation_matrix.csv", index=False)
M.round(3)

,name,model,val_DA,val_BalAcc,val_F1_macro,val_MCC,val_AUC,test_DA,test_BalAcc,test_F1_macro,test_MCC,test_AUC,test_auc_ci95
0,A LR,LR,0.506,0.508,0.504,0.016,0.501,0.517,0.515,0.513,0.031,0.527,NaN
1,A HGB,HGB,0.523,0.521,0.520,0.042,0.503,0.500,0.503,0.498,0.005,0.538,NaN
2,B_1 LR,LR,0.517,0.516,0.516,0.032,0.474,0.517,0.517,0.514,0.034,0.539,NaN
3,B_1 HGB,HGB,0.517,0.516,0.515,0.031,0.471,0.455,0.457,0.453,-0.086,0.491,NaN
4,B_2 LR,LR,0.528,0.531,0.527,0.062,0.529,0.500,0.505,0.499,0.009,0.561,NaN
5,B_2 HGB,HGB,0.591,0.592,0.591,0.184,0.592,0.506,0.518,0.506,0.037,0.507,NaN
6,C (FinBERT art-CLS),HGB,0.591,0.588,0.588,0.179,0.568,0.584,0.561,0.560,0.127,0.594,"[0.515, 0.662]"
